<a href="https://colab.research.google.com/github/va4756/algio_sklearn/blob/main/llm_product_chap5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 5-1 scratchGPT

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 20.9 MB/s eta 0:00:00


In [ ]:
import os
import torch
import torch.nn as nn
from accelerate import Accelerator
import bitsandbytes as bnb

In [ ]:
batch_size = 64
block_size= 256
max_iters = 5000
eval_interval = 500
learning_rate = 3e-4
eval_iters = 200
n_embed = 384
n_head = 6
n_layer = 6
dropout = 0.2
accelerator = Accelerator()
device = accelerator.device
doing_quantization = False

In [ ]:
# dataset -> DeepLearning_pytorch폴더 -> llm-production -> data
with open("/content/crimeandpunishment.txt", "r", encoding="utf-8") as f:
    text = f.read()

In [ ]:
# Character-based pseudo-tokenization
chars = sorted(list(set(text)))
vocab_size = len(chars)
utt2int = {ch: i for i, ch in enumerate(chars)}
int2utt = {i: ch for i, ch in enumerate(chars)}

In [ ]:
# Define the overall GPT Architecture
class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embed)
        self.positional_embedding = nn.Embedding(block_size, n_embed)
        self.blocks = nn.Sequential(
            *[Block(n_embed, n_head=n_head) for _ in range(n_layer)]
        )
        self.ln_f = nn.LayerNorm(n_embed)
        self.lm_head = nn.Linear(n_embed, vocab_size)

        self.apply(self._init_weights)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding(idx)
        pos_emb = self.positional_embedding(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = nn.functional.cross_entropy(logits, targets)

        return logits, loss

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = nn.functional.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

In [ ]:
# Define the building blocks of the model
class Block(nn.Module):
    def __init__(self, n_embed, n_head):
        super().__init__()
        head_size = n_embed // n_head
        self.self_attention = MultiHeadAttention(n_head, head_size)
        self.feed_forward = FeedForward(n_embed)
        self.ln1 = nn.LayerNorm(n_embed)
        self.ln2 = nn.LayerNorm(n_embed)

    def forward(self, x):
        x = x + self.self_attention(self.ln1(x))
        x = x + self.feed_forward(self.ln2(x))
        return x

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList(
            [Head(head_size) for _ in range(num_heads)]
        )
        self.projection = nn.Linear(head_size * num_heads, n_embed)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.projection(out))
        return out

In [ ]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embed, head_size, bias=False)
        self.query = nn.Linear(n_embed, head_size, bias=False)
        self.value = nn.Linear(n_embed, head_size, bias=False)
        self.register_buffer(
            "tril", torch.tril(torch.ones(block_size, block_size))
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        _, T, _ = x.shape
        k = self.key(x)
        q = self.query(x)
        # gpt의 의견
        attention = (
                q @ k.transpose(-2, -1)) / (k.shape[-1] ** 0.5)
        # attention = q @ k.transpose(-2, -1) * k.shape[-1] ** 0.5
        attention = attention.masked_fill(
            self.tril[:T, :T] == 0, float("-inf")
        )
        attention = nn.functional.softmax(attention, dim=-1)
        attention = self.dropout(attention)

        v = self.value(x)
        out = attention @ v
        return out

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, n_embed):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embed, 4 * n_embed),
            nn.ReLU(),
            nn.Linear(4 * n_embed, n_embed),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
# Helper functions for training
def encode(string):
    return [utt2int[c] for c in string]

In [ ]:
def decode(line):
    return "".join([int2utt[i] for i in line])

In [ ]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

In [ ]:
def get_batch(split):
    data = train_data if split == "train" else val_data
    idx = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i : i + block_size] for i in idx])
    y = torch.stack([data[i + 1 : i + block_size + 1] for i in idx])
    x, y = x.to(device), y.to(device)
    return x, y

In [ ]:
model = GPT().to(device)

In [ ]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, y = get_batch(split)
            logits, loss = model(X, y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [ ]:
# Train the model
if __name__ == "__main__":
    model = GPT().to(device)
    print("Instantiated Model")
    print(
        sum(param.numel() for param in model.parameters()) / 1e6,
        "Model parameters",
    )
    optimizer = (
        torch.optim.AdamW(model.parameters(), lr=learning_rate)
        if not doing_quantization else bnb.optim.Adam(model.parameters(), lr=learning_rate)
    )
    print("Instantiated Optimizer")

    model, optimizer, train_data = accelerator.prepare(
        model, optimizer, train_data
    )
    print("Prepared model, optimizer, and data")

    # Training block
    for iter in range(max_iters):
        print(f"Running Epoch {iter}")
        if iter % eval_interval == 0 or iter == max_iters - 1:
            losses = estimate_loss()
            print(f"| step {iter}: train loss {losses['train']:.4f} "
                "| validation loss {losses['val']:.4f} |")

        xb, yb = get_batch("train")
        logits, loss = model(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        accelerator.backward(loss)
        optimizer.step()

    # Create model directory
    model_dir = "./models/scratchGPT/"
    if not os.path.exists(model_dir):
        os.makedirs(model_dir)

    # Save the model
    model_path = model_dir + "model.pt"
    torch.save(model.state_dict(), model_path)

    # Load the saved model
    loaded_model = GPT().to(device)
    loaded_model.load_state_dict(torch.load(model_path))

    # Test the loaded moel
    context = torch.zeros((1, 1), dtype=torch.long, device=device)
    print(
        decode(
            loaded_model.generate(context, max_new_tokens=500)[0].tolist()
        )
    )

Instantiated Model
10.806616 Model parameters
Instantiated Optimizer
Prepared model, optimizer, and data
Running Epoch 0
| step 0: train loss 4.5113 | validation loss {losses['val']:.4f} |
Running Epoch 1
Running Epoch 2
Running Epoch 3
Running Epoch 4
Running Epoch 5
Running Epoch 6
Running Epoch 7
Running Epoch 8
Running Epoch 9
Running Epoch 10
Running Epoch 11
Running Epoch 12
Running Epoch 13
Running Epoch 14
Running Epoch 15
Running Epoch 16
Running Epoch 17
Running Epoch 18
Running Epoch 19
Running Epoch 20
Running Epoch 21
Running Epoch 22
Running Epoch 23
Running Epoch 24
Running Epoch 25
Running Epoch 26
Running Epoch 27
Running Epoch 28
Running Epoch 29
Running Epoch 30
Running Epoch 31
Running Epoch 32
Running Epoch 33
Running Epoch 34
Running Epoch 35
Running Epoch 36
Running Epoch 37
Running Epoch 38
Running Epoch 39
Running Epoch 40
Running Epoch 41
Running Epoch 42
Running Epoch 43
Running Epoch 44
Running Epoch 45
Running Epoch 46
Running Epoch 47
Running Epoch 48
Runn

KeyboardInterrupt: 

# 5-2 finetuning

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    GPT2Config,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer
)
from datasets import load_dataset

In [ ]:
# Load and format the dataset
dataset = load_dataset("text", data_files="/content/crimeandpunishment.txt")
dataset = dataset.filter(lambda sentence: len(sentence["text"]) > 1)
print(dataset["train"][0])

{'text': 'Part I, Chapter I'}


In [ ]:
# Create model directory to save to
model_dir = "./models/betterGPT/"
if not os.path.exists(model_dir):
    os.makedirs(model_dir)

In [ ]:
# Establish our GPT2 parameters (different from the paper and scratchGPT)
config = GPT2Config(vocab_size=50261,
                    n_positions=256,
                    n_embd=768,
                    activation_function="gelu")

In [ ]:
from huggingface_hub import login

login()

In [ ]:
# Instantiate our tokenizer and our special tokens
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
special_tokens_dict = {
    "bos_token": "<BOS>",
    "eos_token": "<EOS>",
    "pad_token": "<PAD>",
    "mask_token": "<MASK>"
}
tokenizer.add_special_tokens(special_tokens_dict)

4

In [ ]:
# Instantiate our model from the config
model = GPT2LMHeadModel.from_pretrained(
    "gpt2", config=config, ignore_mismatched_sizes=True
)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                    | Status     |                                                                                             
-----------------------+------------+---------------------------------------------------------------------------------------------
h.{0...11}.attn.bias   | UNEXPECTED |                                                                                             
transformer.wte.weight | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([50257, 768]) vs model:torch.Size([50261, 768])
transformer.wpe.weight | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([1024, 768]) vs model:torch.Size([256, 768])   

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
# Create a tokenize function
def tokenize(batch):
    return tokenizer(
        str(batch), padding="max_length", truncation=True, max_length=256
    )

In [ ]:
# tokenize our whole dataset (so we never have to do it again)
tokenized_dataset = dataset.map(tokenize, batched=False)
print(f"Tokenized: {tokenized_dataset['train'][0]}")

Map:   0%|          | 0/3944 [00:00<?, ? examples/s]

Tokenized: {'text': 'Part I, Chapter I', 'input_ids': [90, 6, 5239, 10354, 705, 7841, 314, 11, 7006, 314, 6, 92, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259, 50259

In [ ]:
# Create a data collator to format the data for training
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=True, mlm_probability=0.15
)

In [ ]:
# Establish training arguments
train_args = TrainingArguments(
    output_dir=model_dir,
    num_train_epochs=1,
    per_device_train_batch_size=8,
    save_steps=5000,
    save_total_limit=2,
    report_to="none"
)

In [ ]:
# Instantiate the Trainer
trainer = Trainer(
    model=model,
    args=train_args,
    data_collator=data_collator,
    train_dataset=tokenized_dataset["train"]
)

In [ ]:
# Train and save the model
trainer.train()
trainer.save_model(model_dir)
tokenizer.save_pretrained(model_dir)

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./models/betterGPT/tokenizer_config.json',
 './models/betterGPT/tokenizer.json')

In [ ]:
# Load the saved model
model = GPT2LMHeadModel.from_pretrained(model_dir)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [ ]:
# Test the saved model
input = "To be or not"
tokenzied_inputs = tokenizer(input, return_tensors="pt")
out = model.generate(
    input_ids=tokenzied_inputs["input_ids"],
    attention_mask=tokenzied_inputs["attention_mask"],
    max_length=256,
    num_beams=5,
    temperature=0.7,
    top_k=50,
    top_p=0.90,
    no_repeat_ngram_size=2
)
print(tokenizer.decode(out[0], skip_special_tokens=True))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


To be or not '",, the the, and, to, I,},., a, of, in,\', he, you, it, that, was, her, at,.', for, his,!,s, had, as,',ov, on, with, not, is, ", she,?, me, all,;, but, He, be,..., him, my,t, have, what, are, this,...., from,na, But, were, one, there,oln,-,,",ik,ask, out, up, your, who, more, too,?", R,!", so, do,.", they, them, if, went,—,ia, would, now, said, no, could, about, am, an, The, been,rov, see, come, thought, know, though, down,itch, man, again, You, It, here,ih, by, Sonia,um, some,a, looked, like, when, time,I, Raz, we, And, very, can,oun, did, into, go, suddenly, don, only, say


# 5-3 PromptTuning

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    default_data_collator,
    get_linear_schedule_with_warmup,
)
from peft import (
    get_peft_model,
    PromptTuningInit,
    PromptTuningConfig,
    TaskType
)
import torch
from datasets import load_dataset
from torch.utils.data import DataLoader
from tqdm import tqdm

In [ ]:
from datasets import load_dataset
dataset = load_dataset("text", data_files="/content/crimeandpunishment.txt")
# dataset = load_dataset("imdb")
print(dataset["train"].column_names)

Generating train split: 0 examples [00:00, ? examples/s]

['text']


In [ ]:
# Helper function to preprocess text - go ahead and skip to the training
def preprocess_function(examples):
    batch_size = len(examples[text_column])
    inputs = [
        f"{text_column} : {x} Label : " for x in examples[text_column]
    ]
    targets = [str(x) for x in examples[label_column]]
    model_inputs = tokenizer(inputs)
    labels = tokenizer(targets)

    for i in range(batch_size):
        sample_input_ids = model_inputs["input_ids"][i]
        label_input_ids = labels["input_ids"][i] + [tokenizer.pad_token_id]
        model_inputs["input_ids"][i] = sample_input_ids + label_input_ids
        labels["input_ids"][i] = [-100] * len(
            sample_input_ids
        ) + label_input_ids
        model_inputs["attention_mask"][i] = [1] * len(
            model_inputs["input_ids"][i]
        )

    for i in range(batch_size):
        sample_input_ids = model_inputs["input_ids"][i]
        label_input_ids = labels["input_ids"][i]
        model_inputs["input_ids"][i] = [tokenizer.pad_token_id] * (max_length - len(sample_input_ids)) \
                                        + sample_input_ids